## Video Games Sales Analysis

# EDA Framework — Video Game Sales & Steam Data

This notebook will cover the data cleaning process for our tables, how we matched datatypes properly, go into detail of how we merged  our two datasets, and provide exploratory data visualizations that will assist with the decision-making process in either accepting or rejecting our hypotheses.

## Section 1: Loading Tables

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

AttributeError: module 'matplotlib' has no attribute 'colors'

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 243a3700130, execution_count=16 error_before_exec=None error_in_exec=module 'matplotlib' has no attribute 'colors' info=<ExecutionInfo object at 243a36c8b50, raw_cell="import pandas as pd
import numpy as np
import matp.." transformed_cell="import pandas as pd
import numpy as np
import matp.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [4]:
games      = pd.read_csv('Cleaned_Data/games_clean.csv',           parse_dates=['release_date'])
matched    = pd.read_csv('Cleaned_Data/games_vgsales_matched.csv', parse_dates=['release_date'])
categories = pd.read_csv('Cleaned_Data/categories_clean.csv')
tags       = pd.read_csv('Cleaned_Data/tags_clean.csv')
genres     = pd.read_csv('Cleaned_Data/genres_clean.csv')

for label, df in [('games', games), ('matched', matched),
                  ('categories', categories), ('tags', tags), ('genres', genres)]:
    print(f'{label:12s}  {df.shape}')

games         (140082, 8)
matched       (1854, 16)
categories    (522569, 2)
tags          (1744632, 2)
genres        (353339, 2)


## Section 2: Fix Known Data Issues

### Problem A — `owners_range` is a string, not a number
SteamSpy stores ownership as a range like `"500,000 .. 1,000,000"`. We split it into a low and high bound, then take the midpoint for plotting and analysis.

### Problem B — `price` is missing for ~35% of rows
Missing price is not random — it overlaps heavily with free-to-play games. We fill `price = 0.0` where `is_free = 1` and price is null, then flag the rest as unknown so they can be excluded from price-sensitive analysis rather than silently skewing results.

In [5]:
# --- Fix owners_range ---
# Split '500,000 .. 1,000,000' into two numeric columns, then compute midpoint

def parse_owners(df):
    split = df['owners_range'].str.replace(',', '', regex=False).str.split('..', expand=True)
    df = df.copy()
    df['owners_low']  = pd.to_numeric(split[0].str.strip(), errors='coerce').fillna(0).astype(int)
    df['owners_high'] = pd.to_numeric(split[1].str.strip(), errors='coerce').fillna(0).astype(int)
    df['owners_mid']  = ((df['owners_low'] + df['owners_high']) / 2).astype(int)
    return df

games   = parse_owners(games)
matched = parse_owners(matched)

# Quick check
games[['owners_range', 'owners_low', 'owners_high', 'owners_mid']].drop_duplicates('owners_range').head(6)

,owners_range,owners_low,owners_high,owners_mid
0,"10,000,000 .. 20,000,000",0,0,0
1,"5,000,000 .. 10,000,000",0,0,0
4,"2,000,000 .. 5,000,000",0,0,0
8,"1,000,000 .. 2,000,000",0,0,0
9,"0 .. 20,000",0,0,0
12,"500,000 .. 1,000,000",0,0,0


In [6]:
# --- Fix price nulls ---

def fix_price(df):
    df = df.copy()
    df['price_filled'] = df['price']
    # Fill 0 for confirmed free-to-play games
    df.loc[(df['is_free'] == 1) & df['price'].isna(), 'price_filled'] = 0.0
    # Flag paid games where price is still missing
    df['price_unknown'] = df['price_filled'].isna()
    return df

games   = fix_price(games)
matched = fix_price(matched)

for label, df in [('games', games), ('matched', matched)]:
    resolved = ((df['is_free'] == 1) & df['price'].isna()).sum()
    still_null = df['price_unknown'].sum()
    print(f'{label}: {resolved} free-game nulls filled with 0,  {still_null} paid-game prices still unknown')

games: 22345 free-game nulls filled with 0,  26847 paid-game prices still unknown
matched: 61 free-game nulls filled with 0,  155 paid-game prices still unknown


---
## Section 3: Match Quality Audit

Only 1.34% of Steam games matched to a VGChartz entry. Before using `matched` in any hypothesis test, we check how trustworthy those matches are and flag the risky ones.

In [7]:
# Match score distribution
# PURPOSE: Matches below ~87 have meaningful false-positive risk per the quality
# report. This histogram helps decide whether to tighten the threshold before
# any hypothesis test that relies on VGChartz sales figures.

fig, ax = plt.subplots()
ax.hist(matched['vg_match_score'], bins=30, color='steelblue', edgecolor='white')
ax.axvline(83, color='orange', linestyle='--', linewidth=1.5, label='Current threshold (83)')
ax.axvline(90, color='red',    linestyle='--', linewidth=1.5, label='Stricter threshold (90)')
ax.set_title('Steam ↔ VGChartz Fuzzy Match Score Distribution')
ax.set_xlabel('Match Score (0–100)')
ax.set_ylabel('Number of Matched Games')
ax.legend()
plt.tight_layout()
plt.show()

for t in [83, 87, 90, 95]:
    n = (matched['vg_match_score'] >= t).sum()
    print(f'  Threshold >= {t}: {n} matches')

AttributeError: module 'matplotlib' has no attribute 'backends'

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 243f21ad450, execution_count=7 error_before_exec=None error_in_exec=module 'matplotlib' has no attribute 'backends' info=<ExecutionInfo object at 243f21ad6d0, raw_cell="# Match score distribution
# PURPOSE: Matches belo.." transformed_cell="# Match score distribution
# PURPOSE: Matches belo.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [13]:
# Flag edition/version mismatches
# PURPOSE: The quality report found matches where one title includes words like
# 'Remastered' or 'HD' and the other does not — these are different products.
# We flag them so they can be excluded when sales accuracy matters.

VERSION_KEYWORDS = ['remastered', 'hd', 'goty', 'definitive', 'complete',
                    'collection', 'anthology', 'trilogy', 'bundle']

def flag_version_mismatch(steam_name, vg_name):
    s = str(steam_name).lower()
    v = str(vg_name).lower()
    for kw in VERSION_KEYWORDS:
        if (kw in s) != (kw in v):   # keyword on one side only
            return True
    return False

matched['flag_version_mismatch'] = matched.apply(
    lambda r: flag_version_mismatch(r['name'], r['vg_name']), axis=1
)
matched['flag_low_score'] = matched['vg_match_score'] < 87
matched['flag_any']       = matched['flag_version_mismatch'] | matched['flag_low_score']

print(f"Version mismatch flags : {matched['flag_version_mismatch'].sum()}")
print(f"Low score flags        : {matched['flag_low_score'].sum()}")
print(f"Total flagged          : {matched['flag_any'].sum()}")
print(f"Clean matches left     : {(~matched['flag_any']).sum()}")

Version mismatch flags : 20
Low score flags        : 482
Total flagged          : 493
Clean matches left     : 1361
Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 2439d4f6a50, execution_count=13 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 2439d877950, raw_cell="# Flag edition/version mismatches
# PURPOSE: The q.." transformed_cell="# Flag edition/version mismatches
# PURPOSE: The q.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [14]:
# Are matched games representative of the full catalog?
# PURPOSE: If matched games are systematically older or more popular than average,
# conclusions drawn from them cannot generalize to all Steam games. This is an
# important limitation to acknowledge in your write-up.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Release year
for df, label, color in [(games, 'All Steam', 'steelblue'), (matched, 'Matched only', 'tomato')]:
    years = df['release_date'].dt.year
    years = years[years.between(2000, 2024)]
    axes[0].hist(years, bins=24, alpha=0.6, label=label, color=color, density=True)
axes[0].set_title('Release Year: All Steam vs. Matched Subset')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Density')
axes[0].legend()

# Ownership midpoint (log scale)
for df, label, color in [(games, 'All Steam', 'steelblue'), (matched, 'Matched only', 'tomato')]:
    axes[1].hist(np.log10(df['owners_mid'].replace(0, 1)), bins=30,
                 alpha=0.6, label=label, color=color, density=True)
axes[1].set_title('Ownership (log₁₀): All Steam vs. Matched Subset')
axes[1].set_xlabel('log₁₀(owners_mid)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'colors'

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 2439d58c2f0, execution_count=14 error_before_exec=None error_in_exec=module 'matplotlib' has no attribute 'colors' info=<ExecutionInfo object at 2439e4146d0, raw_cell="# Are matched games representative of the full cat.." transformed_cell="# Are matched games representative of the full cat.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

## EDA

In [15]:
# Games released per year
# PURPOSE: Most Steam games are post-2014. DLC culture, F2P norms, and the
# Battlefront II controversy are all post-2010 events. Knowing where the bulk
# of our data falls contextualizes every time-based comparison we make.

year_counts = (
    games['release_date'].dt.year
    .value_counts()
    .sort_index()
)
year_counts = year_counts[year_counts.index.between(2000, 2024)]

fig, ax = plt.subplots()
ax.bar(year_counts.index, year_counts.values, color='steelblue', edgecolor='white')
ax.axvline(2017, color='red', linestyle='--', linewidth=1.2, label='Nov 2017 (Battlefront II)')
ax.set_title('Steam Games Released Per Year')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Games')
ax.legend()
plt.tight_layout()
plt.show()

AttributeError: Can only use .dt accessor with datetimelike values

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 243a2314ec0, execution_count=15 error_before_exec=None error_in_exec=Can only use .dt accessor with datetimelike values info=<ExecutionInfo object at 243a1c1ec50, raw_cell="# Games released per year
# PURPOSE: Most Steam ga.." transformed_cell="# Games released per year
# PURPOSE: Most Steam ga.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [17]:
# Free-to-play vs. paid split
# PURPOSE: H3 directly compares F2P to paid games. These counts confirm we have
# enough games in each group, and shows how large the 'price unknown' group is
# so we can decide whether to include or exclude it.

def monetization_label(row):
    if row['is_free'] == 1:
        return 'Free to Play'
    elif row['price_unknown']:
        return 'Price Unknown'
    else:
        return 'Paid'

games['monetization_basic'] = games.apply(monetization_label, axis=1)
counts = games['monetization_basic'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind='barh', ax=ax, color=sns.color_palette('muted', len(counts)))
ax.set_title('Steam Catalog by Monetization Type')
ax.set_xlabel('Number of Games')
for i, v in enumerate(counts):
    ax.text(v + 200, i, f'{v:,}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'colors'

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 243a378cd00, execution_count=17 error_before_exec=None error_in_exec=module 'matplotlib' has no attribute 'colors' info=<ExecutionInfo object at 243a36f5ed0, raw_cell="# Free-to-play vs. paid split
# PURPOSE: H3 direct.." transformed_cell="# Free-to-play vs. paid split
# PURPOSE: H3 direct.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [ ]:
# Top 20 genres and tags by frequency
# PURPOSE: Shows which categories are large enough for statistically valid
# comparisons. Genres or tags with very few games will be excluded from
# hypothesis tests that require a minimum sample size per group.

genre_freq = genres['genre'].value_counts().head(20)
tag_freq   = tags['tag'].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

genre_freq.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 Steam Genres by Frequency')
axes[0].set_xlabel('Number of App-Genre Rows')

tag_freq.sort_values().plot(kind='barh', ax=axes[1], color='mediumseagreen')
axes[1].set_title('Top 20 Steam Tags by Frequency')
axes[1].set_xlabel('Number of App-Tag Rows')

plt.tight_layout()
plt.show()

In [18]:
# Ownership distribution
# PURPOSE: Ownership follows a power law — a tiny fraction of games have
# millions of owners while most have under 20,000. The log-scale version
# on the right is what we'll use for any ownership-based comparisons to
# avoid one blockbuster title distorting group medians.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(games['owners_mid'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Ownership Distribution (linear scale)')
axes[0].set_xlabel('Estimated Owners (midpoint)')
axes[0].set_ylabel('Number of Games')

log_owners = np.log10(games['owners_mid'].replace(0, 1))
axes[1].hist(log_owners, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Ownership Distribution (log₁₀ scale)')
axes[1].set_xlabel('log₁₀(owners_mid)')
axes[1].set_ylabel('Number of Games')
axes[1].set_xticks(range(0, 8))
axes[1].set_xticklabels([f'10^{i}' for i in range(0, 8)])

plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'colors'

Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x000002439C968720> (for post_run_cell), with arguments args (<ExecutionResult object at 2439d6bb620, execution_count=18 error_before_exec=None error_in_exec=module 'matplotlib' has no attribute 'colors' info=<ExecutionInfo object at 243a290c8d0, raw_cell="# Ownership distribution
# PURPOSE: Ownership foll.." transformed_cell="# Ownership distribution
# PURPOSE: Ownership foll.." store_history=True silent=False shell_futures=True cell_id=None> result=None>,),kwargs {}:


AttributeError: module 'matplotlib' has no attribute 'backends'

In [ ]:
# In-App Purchases tag prevalence over time
# PURPOSE: This is the most direct precursor to H2. We check whether 'In-App
# Purchases' appears more or less frequently in games released after Nov 2017.
# If the industry leaned harder into IAP post-controversy, that context matters
# when interpreting whether review scores went up or down.

iap_apps = categories[categories['category'] == 'In-App Purchases']['app_id']

games['has_iap']  = games['app_id'].isin(iap_apps)
games['year']     = games['release_date'].dt.year

iap_by_year = (
    games[games['year'].between(2010, 2024)]
    .groupby('year')['has_iap']
    .agg(count='sum', total='count')
    .assign(pct=lambda d: d['count'] / d['total'] * 100)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(iap_by_year['year'], iap_by_year['count'], color='coral', edgecolor='white')
axes[0].set_title('Games with In-App Purchases — Count per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Games')

axes[1].plot(iap_by_year['year'], iap_by_year['pct'], marker='o', color='coral')
axes[1].set_title('Games with In-App Purchases — % of Releases That Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('% of Games Released')

for ax in axes:
    ax.axvline(2017, color='red', linestyle='--', linewidth=1.2, label='Nov 2017')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# VGSales matched subset — sales by platform and top titles
# PURPOSE: Confirms what the matched 1,854 games actually look like.
# Since this is a PC-heavy platform (Steam), we expect most VGChartz matches
# to be PC titles. The top-sellers chart gives a sanity check that recognizable
# games are coming through, not just noise matches.

clean_matched = matched[~matched['flag_any']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

platform_sales = (
    clean_matched.groupby('vg_platform')['vg_global_sales']
    .sum()
    .sort_values(ascending=True)
    .tail(15)
)
platform_sales.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('VGChartz Global Sales by Platform\n(clean matches only)')
axes[0].set_xlabel('Total Global Sales (millions units)')

top20 = (
    clean_matched.nlargest(20, 'vg_global_sales')[['name', 'vg_global_sales']]
    .sort_values('vg_global_sales')
)
axes[1].barh(top20['name'], top20['vg_global_sales'], color='mediumseagreen')
axes[1].set_title('Top 20 Matched Games by VGChartz Global Sales')
axes[1].set_xlabel('Global Sales (millions units)')

plt.tight_layout()
plt.show()

---
## Section 5: Pre-Hypothesis Subset Checks

Before running any statistical tests, confirm the sample size of each group your hypotheses depend on. If a group is too small, you'll know now rather than mid-analysis.

In [ ]:
# H1 — DLC count groups
# Requires a dlc_count column. If yours is named differently, update it here.

if 'dlc_count' in games.columns:
    h1_heavy  = games[games['dlc_count'] >= 5]
    h1_no_dlc = games[games['dlc_count'] == 0]
    print(f'H1 — games with 5+ DLC : {len(h1_heavy):,}')
    print(f'H1 — games with 0 DLC  : {len(h1_no_dlc):,}')

    fig, ax = plt.subplots(figsize=(10, 4))
    games['dlc_count'].clip(upper=20).dropna().hist(bins=21, ax=ax,
                                                     color='steelblue', edgecolor='white')
    ax.axvline(5, color='red', linestyle='--', label='H1 threshold (5+)')
    ax.set_title('DLC Count Distribution (capped at 20)')
    ax.set_xlabel('Number of DLC Packs')
    ax.set_ylabel('Number of Games')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('[!] dlc_count column not found. Add it before running H1.')

In [ ]:
# H2 — Pre vs. post Battlefront II, for IAP-tagged games

BATTLEFRONT = pd.Timestamp('2017-11-01')

h2_base = games[games['has_iap']].dropna(subset=['release_date'])
h2_pre  = h2_base[h2_base['release_date'] <  BATTLEFRONT]
h2_post = h2_base[h2_base['release_date'] >= BATTLEFRONT]

print(f'H2 — IAP games pre-Nov 2017:  {len(h2_pre):,}')
print(f'H2 — IAP games post-Nov 2017: {len(h2_post):,}')

In [ ]:
# H3 — F2P vs. paid groups
# Requires a review_score column. Update the name below to match your actual column.

REVIEW_COL = 'review_score'   # <-- update if named differently

if REVIEW_COL in games.columns:
    h3_base = games.dropna(subset=[REVIEW_COL])
    h3_paid = h3_base[h3_base['is_free'] == 0]
    h3_f2p  = h3_base[h3_base['is_free'] == 1]
    print(f'H3 — paid games with review score : {len(h3_paid):,}')
    print(f'H3 — F2P games with review score  : {len(h3_f2p):,}')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(h3_paid[REVIEW_COL].dropna(), bins=30, alpha=0.6,
            color='steelblue', label='Paid', edgecolor='white')
    ax.hist(h3_f2p[REVIEW_COL].dropna(),  bins=30, alpha=0.6,
            color='tomato',    label='Free to Play', edgecolor='white')
    ax.set_title('H3 Preview: Review Score Distribution — Paid vs. Free to Play')
    ax.set_xlabel('Review Score')
    ax.set_ylabel('Number of Games')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'[!] Column "{REVIEW_COL}" not found. Update REVIEW_COL above.')

---
## Section 6: Summary Counts

A quick reference table of key figures to cite in your write-up and presentation.

In [ ]:
summary = {
    'Total Steam games':                  len(games),
    'Games with release date':            games['release_date'].notna().sum(),
    'Free-to-play games':                 (games['is_free'] == 1).sum(),
    'Paid games (price known)':           (~games['price_unknown'] & (games['is_free'] == 0)).sum(),
    'Paid games (price unknown)':         (games['price_unknown']  & (games['is_free'] == 0)).sum(),
    'Games with IAP category tag':        games['has_iap'].sum(),
    'Matched to VGChartz (any)':          len(matched),
    'Matched to VGChartz (clean only)':   (~matched['flag_any']).sum(),
}

for k, v in summary.items():
    print(f'  {k:<42s}: {v:>8,}')